In [3]:
import pandas as pd
from pathlib import Path

# Current working directory
BASE = Path.cwd()

train_file = BASE / "Level2_Train_Pair_Disjoint.csv"
test_file  = BASE / "Level2_Test_Pair_Disjoint.csv"

train = pd.read_csv(train_file)
test  = pd.read_csv(test_file)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTRAIN COLUMNS:")
print(train.columns.tolist())

print("\nTEST COLUMNS:")
print(test.columns.tolist())

print("\nFirst 3 rows of TRAIN:")
display(train.head(3))

print("\nFirst 3 rows of TEST:")
display(test.head(3))

Train shape: (176347, 8)
Test shape : (44087, 8)

TRAIN COLUMNS:
['Drug1', 'Drug2', 'Interaction_ID', 'SMILES1', 'SMILES2', 'Level1', 'Level1_binary', 'Pair_ID']

TEST COLUMNS:
['Drug1', 'Drug2', 'Interaction_ID', 'SMILES1', 'SMILES2', 'Level1', 'Level1_binary', 'Pair_ID']

First 3 rows of TRAIN:


,Drug1,Drug2,Interaction_ID,SMILES1,SMILES2,Level1,Level1_binary,Pair_ID
0,DB13231,DB00808,2,[Ca++].CC(O)C([O-])=O.CC(O)C([O-])=O,CC1CC2=CC=CC=C2N1NC(=O)C1=CC(=C(Cl)C=C1)S(N)(=...,2,0,DB00808__DB13231
1,DB00684,DB00533,2,NC[C@H]1O[C@H](O[C@@H]2[C@@H](N)C[C@@H](N)[C@H...,CS(=O)(=O)C1=CC=C(C=C1)C1=C(C(=O)OC1)C1=CC=CC=C1,2,0,DB00533__DB00684
2,DB00955,DB00768,2,CCN[C@@H]1C[C@H](N)[C@@H](O[C@H]2OC(CN)=CC[C@H...,CN(C)CC\C=C1\C2=CC=CC=C2COC2=C1C=C(CC(O)=O)C=C2,2,0,DB00768__DB00955



First 3 rows of TEST:


,Drug1,Drug2,Interaction_ID,SMILES1,SMILES2,Level1,Level1_binary,Pair_ID
0,DB00865,DB13209,2,C[C@@H](CC1=CC=CC=C1)N(C)CC1=CC=CC=C1,[OH-].[OH-].[OH-].[OH-].[OH-].[OH-].[OH-].[OH-...,2,0,DB00865__DB13209
1,DB02659,DB00570,2,[H][C@@](C)(CCC(O)=O)[C@@]1([H])CC[C@@]2([H])[...,[H][C@@]12N(C)C3=CC(OC)=C(C=C3[C@@]11CCN3CC=C[...,2,0,DB00570__DB02659
2,DB00563,DB01202,2,CN(CC1=CN=C2N=C(N)N=C(N)C2=N1)C1=CC=C(C=C1)C(=...,CC[C@H](N1CCCC1=O)C(N)=O,2,0,DB00563__DB01202


In [5]:
# ============================================================
# STEP 2 — Drug identity overlap in current pair-disjoint split
# ============================================================

# Unique drug identities in each partition
train_drugs = set(train["Drug1"]).union(set(train["Drug2"]))
test_drugs  = set(test["Drug1"]).union(set(test["Drug2"]))

# Shared and exclusive drugs
shared_drugs = train_drugs.intersection(test_drugs)
train_only_drugs = train_drugs - test_drugs
test_only_drugs  = test_drugs - train_drugs

print("=== DRUG IDENTITY OVERLAP AUDIT ===")
print(f"Unique drugs in Train : {len(train_drugs):,}")
print(f"Unique drugs in Test  : {len(test_drugs):,}")
print(f"Shared drugs           : {len(shared_drugs):,}")
print(f"Train-only drugs       : {len(train_only_drugs):,}")
print(f"Test-only drugs        : {len(test_only_drugs):,}")

print("\n=== OVERLAP PERCENTAGES ===")

train_overlap_pct = 100 * len(shared_drugs) / len(train_drugs)
test_overlap_pct  = 100 * len(shared_drugs) / len(test_drugs)

print(f"Train drugs also appearing in Test : {train_overlap_pct:.2f}%")
print(f"Test drugs already seen in Train    : {test_overlap_pct:.2f}%")

# ------------------------------------------------------------
# Classify TEST PAIRS according to drug novelty
# ------------------------------------------------------------

def novelty_status(row):
    d1_seen = row["Drug1"] in train_drugs
    d2_seen = row["Drug2"] in train_drugs

    if d1_seen and d2_seen:
        return "Both_seen"
    elif d1_seen or d2_seen:
        return "One_unseen"
    else:
        return "Both_unseen"

test_audit = test.copy()
test_audit["Drug_novelty"] = test_audit.apply(novelty_status, axis=1)

print("\n=== TEST PAIR NOVELTY ===")

counts = test_audit["Drug_novelty"].value_counts()
percentages = test_audit["Drug_novelty"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "N_pairs": counts,
    "Percentage": percentages
})

display(summary)

# Save for later use
test_audit.to_csv(
    BASE / "Reviewer1_Current_Test_Drug_Overlap_Audit.csv",
    index=False
)

print("\nSaved: Reviewer1_Current_Test_Drug_Overlap_Audit.csv")

=== DRUG IDENTITY OVERLAP AUDIT ===
Unique drugs in Train : 1,829
Unique drugs in Test  : 1,733
Shared drugs           : 1,717
Train-only drugs       : 112
Test-only drugs        : 16

=== OVERLAP PERCENTAGES ===
Train drugs also appearing in Test : 93.88%
Test drugs already seen in Train    : 99.08%

=== TEST PAIR NOVELTY ===


,N_pairs,Percentage
Drug_novelty,,
Both_seen,44070,99.96144
One_unseen,17,0.03856



Saved: Reviewer1_Current_Test_Drug_Overlap_Audit.csv


In [6]:
# ============================================================
# STEP 3 — Inspect full Level 2 dataset before drug-based splits
# ============================================================

# Reconstruct the complete current Level 2 dataset
full_data = pd.concat([train, test], ignore_index=True)

print("=== FULL LEVEL 2 DATASET ===")
print(f"Total records       : {len(full_data):,}")
print(f"Unique Pair_ID      : {full_data['Pair_ID'].nunique():,}")
print(f"Unique Drug1        : {full_data['Drug1'].nunique():,}")
print(f"Unique Drug2        : {full_data['Drug2'].nunique():,}")

all_drugs = set(full_data["Drug1"]).union(set(full_data["Drug2"]))
print(f"Unique drugs overall: {len(all_drugs):,}")

print("\n=== TARGET INFORMATION ===")
print(f"Unique Interaction_ID classes: {full_data['Interaction_ID'].nunique()}")
print(f"Unique Level1 values          : {full_data['Level1'].nunique()}")

print("\nInteraction_ID distribution:")
class_counts = (
    full_data["Interaction_ID"]
    .value_counts()
    .sort_index()
)

display(class_counts.to_frame("N"))

print("\n=== PAIR DUPLICATION CHECK ===")
print(
    "Duplicated Pair_ID records:",
    full_data["Pair_ID"].duplicated().sum()
)

print("\n=== DRUG FREQUENCY ===")

drug_frequency = pd.concat(
    [full_data["Drug1"], full_data["Drug2"]]
).value_counts()

print("Min interactions per drug   :", drug_frequency.min())
print("Median interactions per drug:", drug_frequency.median())
print("Mean interactions per drug  :", round(drug_frequency.mean(), 2))
print("Max interactions per drug   :", drug_frequency.max())

print("\n10 most frequent drugs:")
display(drug_frequency.head(10).to_frame("N_interactions"))

=== FULL LEVEL 2 DATASET ===
Total records       : 220,434
Unique Pair_ID      : 219,941
Unique Drug1        : 1,765
Unique Drug2        : 1,754
Unique drugs overall: 1,845

=== TARGET INFORMATION ===
Unique Interaction_ID classes: 61
Unique Level1 values          : 2

Interaction_ID distribution:


,N
Interaction_ID,
2,1300
3,27258
5,796
6,543
10,2819
...,...
105,834
108,1825
109,1452



=== PAIR DUPLICATION CHECK ===
Duplicated Pair_ID records: 493

=== DRUG FREQUENCY ===
Min interactions per drug   : 1
Median interactions per drug: 185.0
Mean interactions per drug  : 238.95
Max interactions per drug   : 1028

10 most frequent drugs:


,N_interactions
DB01118,1028
DB01149,988
DB01174,980
DB00285,966
DB00794,952
DB00564,930
DB00176,930
DB00252,929
DB11672,926
DB00246,918


In [7]:
# ============================================================
# STEP 4 — Audit duplicated Pair_ID / multilabel pairs
# ============================================================

# All records belonging to duplicated Pair_IDs
dup_mask = full_data["Pair_ID"].duplicated(keep=False)
dup_data = full_data.loc[dup_mask].copy()

print("=== DUPLICATED PAIR AUDIT ===")

print(f"Records belonging to duplicated Pair_IDs : {len(dup_data):,}")
print(f"Unique duplicated Pair_IDs               : {dup_data['Pair_ID'].nunique():,}")
print(f"Extra records beyond one per Pair_ID      : {len(full_data) - full_data['Pair_ID'].nunique():,}")

# Number of distinct subtype labels associated with each pair
labels_per_pair = (
    full_data.groupby("Pair_ID")["Interaction_ID"]
    .nunique()
    .sort_values(ascending=False)
)

multi_label_pairs = labels_per_pair[labels_per_pair > 1]

print("\n=== TRUE MULTI-LABEL PAIRS ===")
print(f"Pairs with >1 distinct Interaction_ID : {len(multi_label_pairs):,}")

print("\nNumber of distinct labels per multi-label pair:")
display(
    multi_label_pairs.value_counts()
    .sort_index()
    .rename_axis("N_distinct_labels")
    .to_frame("N_pairs")
)

# Pairs duplicated but carrying exactly the same Interaction_ID
same_label_duplicates = labels_per_pair[
    (full_data.groupby("Pair_ID").size() > 1) &
    (labels_per_pair == 1)
]

print("\n=== SAME-LABEL DUPLICATES ===")
print(f"Duplicated pairs with only one Interaction_ID: {len(same_label_duplicates):,}")

# Inspect examples of genuine multi-label pairs
if len(multi_label_pairs) > 0:
    example_ids = multi_label_pairs.index[:10]

    examples = (
        full_data[full_data["Pair_ID"].isin(example_ids)]
        [["Pair_ID", "Drug1", "Drug2", "Interaction_ID", "Level1", "Level1_binary"]]
        .sort_values(["Pair_ID", "Interaction_ID"])
    )

    print("\nExamples of multi-label pairs:")
    display(examples)

# Check whether multiple labels cross PK/PD branches
branch_per_pair = (
    full_data.groupby("Pair_ID")["Level1"]
    .nunique()
)

cross_branch_pairs = branch_per_pair[branch_per_pair > 1]

print("\n=== PK/PD CROSS-BRANCH CHECK ===")
print(f"Pairs associated with >1 Level1 branch: {len(cross_branch_pairs):,}")

=== DUPLICATED PAIR AUDIT ===
Records belonging to duplicated Pair_IDs : 986
Unique duplicated Pair_IDs               : 493
Extra records beyond one per Pair_ID      : 493

=== TRUE MULTI-LABEL PAIRS ===
Pairs with >1 distinct Interaction_ID : 423

Number of distinct labels per multi-label pair:


,N_pairs
N_distinct_labels,
2,423



=== SAME-LABEL DUPLICATES ===
Duplicated pairs with only one Interaction_ID: 70

Examples of multi-label pairs:


,Pair_ID,Drug1,Drug2,Interaction_ID,Level1,Level1_binary
20973,DB00196__DB00997,DB00997,DB00196,3,2,0
116729,DB00196__DB00997,DB00997,DB00196,89,2,0
185904,DB00238__DB00401,DB00238,DB00401,20,2,0
197328,DB00238__DB00401,DB00238,DB00401,104,2,0
34389,DB00312__DB00401,DB00312,DB00401,20,2,0
85617,DB00312__DB00401,DB00312,DB00401,104,2,0
27510,DB00343__DB01219,DB00343,DB01219,12,2,0
104437,DB00343__DB01219,DB01219,DB00343,89,2,0
19171,DB00440__DB09027,DB09027,DB00440,3,2,0
36772,DB00440__DB09027,DB00440,DB09027,20,2,0



=== PK/PD CROSS-BRANCH CHECK ===
Pairs associated with >1 Level1 branch: 60


In [8]:
# ============================================================
# STEP 5 — Detailed audit of the 423 multi-label pairs
# ============================================================

# Get all genuine multi-label Pair_IDs
multi_ids = set(multi_label_pairs.index)

multi = full_data[
    full_data["Pair_ID"].isin(multi_ids)
].copy()

# Build one row per Pair_ID
multi_summary = (
    multi.groupby("Pair_ID")
    .agg(
        Drug1=("Drug1", "first"),
        Drug2=("Drug2", "first"),
        N_subtypes=("Interaction_ID", "nunique"),
        Subtypes=("Interaction_ID",
                  lambda x: " | ".join(map(str, sorted(set(x))))),
        N_branches=("Level1", "nunique"),
        Branches=("Level1",
                  lambda x: " | ".join(map(str, sorted(set(x)))))
    )
    .reset_index()
)

multi_summary["Type"] = multi_summary["N_branches"].map(
    {1: "Same_branch_multilabel",
     2: "Cross_branch_multilabel"}
)

print("=== MULTI-LABEL SUMMARY ===")
print(f"Total multi-label pairs       : {len(multi_summary):,}")
print(
    f"Same-branch multi-label pairs : "
    f"{(multi_summary['Type']=='Same_branch_multilabel').sum():,}"
)
print(
    f"Cross-branch multi-label pairs: "
    f"{(multi_summary['Type']=='Cross_branch_multilabel').sum():,}"
)

print("\nPercent of unique Level 2 pairs:")
print(
    f"{100 * len(multi_summary) / full_data['Pair_ID'].nunique():.4f}%"
)

print("\nExamples of cross-branch pairs:")
display(
    multi_summary[
        multi_summary["Type"] == "Cross_branch_multilabel"
    ].head(20)
)

# Save complete audit
multi_summary.to_csv(
    BASE / "Reviewer1_423_Multilabel_Pairs_Audit.csv",
    index=False
)

print("\nSaved: Reviewer1_423_Multilabel_Pairs_Audit.csv")

=== MULTI-LABEL SUMMARY ===
Total multi-label pairs       : 423
Same-branch multi-label pairs : 363
Cross-branch multi-label pairs: 60

Percent of unique Level 2 pairs:
0.1923%

Examples of cross-branch pairs:


,Pair_ID,Drug1,Drug2,N_subtypes,Subtypes,N_branches,Branches,Type
6,DB00182__DB01156,DB00182,DB01156,2,89 | 113,2,2 | 3,Cross_branch_multilabel
52,DB00246__DB01233,DB00246,DB01233,2,53 | 113,2,2 | 3,Cross_branch_multilabel
73,DB00305__DB00541,DB00541,DB00305,2,17 | 113,2,2 | 3,Cross_branch_multilabel
74,DB00307__DB01229,DB00307,DB01229,2,3 | 113,2,2 | 3,Cross_branch_multilabel
82,DB00320__DB00541,DB00541,DB00320,2,17 | 113,2,2 | 3,Cross_branch_multilabel
83,DB00320__DB00997,DB00997,DB00320,2,3 | 113,2,2 | 3,Cross_branch_multilabel
92,DB00363__DB00997,DB00997,DB00363,2,3 | 113,2,2 | 3,Cross_branch_multilabel
93,DB00363__DB01233,DB01233,DB00363,2,89 | 113,2,2 | 3,Cross_branch_multilabel
126,DB00420__DB01233,DB01233,DB00420,2,89 | 113,2,2 | 3,Cross_branch_multilabel
139,DB00477__DB01233,DB01233,DB00477,2,89 | 113,2,2 | 3,Cross_branch_multilabel



Saved: Reviewer1_423_Multilabel_Pairs_Audit.csv


In [9]:
# ============================================================
# STEP 6 — Build graph statistics for drug-disjoint splitting
# ============================================================

import numpy as np

# Work at unique canonical-pair level for split design
# For now keep one row per Pair_ID only for graph structure.
pair_graph = (
    full_data[
        ["Pair_ID", "Drug1", "Drug2"]
    ]
    .drop_duplicates("Pair_ID")
    .copy()
)

print("=== PAIR GRAPH ===")
print(f"Unique pairs : {len(pair_graph):,}")

all_drugs = sorted(
    set(pair_graph["Drug1"]).union(set(pair_graph["Drug2"]))
)

print(f"Unique drugs : {len(all_drugs):,}")

# Degree = number of unique pairs containing each drug
degree = pd.concat(
    [pair_graph["Drug1"], pair_graph["Drug2"]]
).value_counts()

print("\n=== DRUG DEGREE DISTRIBUTION ===")
print(f"Minimum degree : {degree.min():,}")
print(f"25th percentile: {degree.quantile(0.25):.1f}")
print(f"Median degree  : {degree.median():.1f}")
print(f"75th percentile: {degree.quantile(0.75):.1f}")
print(f"Maximum degree : {degree.max():,}")

# How many drugs are rare / highly connected?
thresholds = [1, 2, 5, 10, 20, 50, 100, 200, 500]

print("\n=== NUMBER OF DRUGS BY MAXIMUM DEGREE ===")
for t in thresholds:
    print(
        f"Degree <= {t:3d}: "
        f"{(degree <= t).sum():4d} drugs"
    )

print("\n=== MOST CONNECTED DRUGS ===")
display(
    degree.head(20).to_frame("N_unique_pairs")
)

# Basic connected-components check
try:
    import networkx as nx

    G = nx.Graph()
    G.add_edges_from(
        pair_graph[["Drug1", "Drug2"]]
        .itertuples(index=False, name=None)
    )

    components = sorted(
        [len(c) for c in nx.connected_components(G)],
        reverse=True
    )

    print("\n=== CONNECTED COMPONENTS ===")
    print(f"Number of components : {len(components):,}")
    print(f"Largest component    : {components[0]:,} drugs")
    print(f"Second largest       : {components[1] if len(components) > 1 else 0:,} drugs")
    print(f"Isolated components of size 2: {sum(x == 2 for x in components):,}")

except ImportError:
    print("\nnetworkx not installed — connected component check skipped.")

=== PAIR GRAPH ===
Unique pairs : 219,941
Unique drugs : 1,845

=== DRUG DEGREE DISTRIBUTION ===
Minimum degree : 1
25th percentile: 53.0
Median degree  : 185.0
75th percentile: 375.0
Maximum degree : 1,021

=== NUMBER OF DRUGS BY MAXIMUM DEGREE ===
Degree <=   1:   77 drugs
Degree <=   2:  109 drugs
Degree <=   5:  155 drugs
Degree <=  10:  220 drugs
Degree <=  20:  319 drugs
Degree <=  50:  453 drugs
Degree <= 100:  616 drugs
Degree <= 200:  954 drugs
Degree <= 500: 1628 drugs

=== MOST CONNECTED DRUGS ===


,N_unique_pairs
DB01118,1021
DB01149,984
DB01174,972
DB00285,965
DB00794,944
DB00176,929
DB11672,922
DB00252,922
DB00246,916
DB00564,914



=== CONNECTED COMPONENTS ===
Number of components : 2
Largest component    : 1,843 drugs
Second largest       : 2 drugs
Isolated components of size 2: 1


In [10]:
# ============================================================
# STEP 7 — Feasibility of drug-based holdout
# ============================================================

import numpy as np
import pandas as pd

# Use unique canonical pairs only for feasibility analysis
pairs_unique = (
    full_data[
        ["Pair_ID", "Drug1", "Drug2", "Interaction_ID"]
    ]
    .drop_duplicates("Pair_ID")
    .copy()
)

all_drugs_array = np.array(
    sorted(set(pairs_unique["Drug1"]).union(pairs_unique["Drug2"]))
)

fractions = [0.10, 0.15, 0.20, 0.25, 0.30]
seeds = [11, 22, 33, 44, 55]

results = []

for frac in fractions:
    for seed in seeds:

        rng = np.random.default_rng(seed)

        n_unseen = int(round(len(all_drugs_array) * frac))

        unseen = set(
            rng.choice(
                all_drugs_array,
                size=n_unseen,
                replace=False
            )
        )

        d1_unseen = pairs_unique["Drug1"].isin(unseen)
        d2_unseen = pairs_unique["Drug2"].isin(unseen)

        # Neither drug unseen -> eligible training pair
        train_mask = (~d1_unseen) & (~d2_unseen)

        # Exactly one unseen drug
        one_unseen_mask = d1_unseen ^ d2_unseen

        # Both drugs unseen
        both_unseen_mask = d1_unseen & d2_unseen

        train_tmp = pairs_unique[train_mask]
        one_tmp = pairs_unique[one_unseen_mask]
        both_tmp = pairs_unique[both_unseen_mask]

        results.append({
            "Unseen_fraction": frac,
            "Seed": seed,
            "N_unseen_drugs": n_unseen,

            "Train_pairs": len(train_tmp),
            "One_unseen_pairs": len(one_tmp),
            "Both_unseen_pairs": len(both_tmp),

            "Train_classes": train_tmp["Interaction_ID"].nunique(),
            "One_unseen_classes": one_tmp["Interaction_ID"].nunique(),
            "Both_unseen_classes": both_tmp["Interaction_ID"].nunique()
        })

feasibility = pd.DataFrame(results)

print("=== DRUG-HOLDOUT FEASIBILITY ===")
display(feasibility)

print("\n=== MEAN RESULTS BY HOLDOUT FRACTION ===")

summary = (
    feasibility
    .groupby("Unseen_fraction")
    .agg(
        Mean_train_pairs=("Train_pairs", "mean"),
        Mean_one_unseen_pairs=("One_unseen_pairs", "mean"),
        Mean_both_unseen_pairs=("Both_unseen_pairs", "mean"),

        Min_train_classes=("Train_classes", "min"),
        Min_one_unseen_classes=("One_unseen_classes", "min"),
        Min_both_unseen_classes=("Both_unseen_classes", "min")
    )
)

display(summary)

feasibility.to_csv(
    BASE / "Reviewer1_Drug_Holdout_Feasibility.csv",
    index=False
)

print("\nSaved: Reviewer1_Drug_Holdout_Feasibility.csv")

=== DRUG-HOLDOUT FEASIBILITY ===


,Unseen_fraction,Seed,N_unseen_drugs,Train_pairs,One_unseen_pairs,Both_unseen_pairs,Train_classes,One_unseen_classes,Both_unseen_classes
0,0.10,11,184,177493,40195,2253,61,61,46
1,0.10,22,184,179191,38772,1978,61,60,42
2,0.10,33,184,171699,45332,2910,61,61,42
3,0.10,44,184,176164,41360,2417,61,60,44
4,0.10,55,184,177216,40428,2297,61,61,45
5,0.15,11,277,157697,57140,5104,61,61,52
6,0.15,22,277,163101,52639,4201,61,61,46
7,0.15,33,277,159868,55345,4728,61,61,50
8,0.15,44,277,153541,60436,5964,58,61,53
9,0.15,55,277,158060,56848,5033,60,61,47



=== MEAN RESULTS BY HOLDOUT FRACTION ===


,Mean_train_pairs,Mean_one_unseen_pairs,Mean_both_unseen_pairs,Min_train_classes,Min_one_unseen_classes,Min_both_unseen_classes
Unseen_fraction,,,,,,
0.10,176352.6,41217.4,2371.0,61,60,42
0.15,158453.4,56481.6,5006.0,58,61,46
0.20,142070.8,69467.8,8402.4,59,61,52
0.25,123871.2,82350.4,13719.4,59,61,55
0.30,106352.4,93291.8,20296.8,58,61,56



Saved: Reviewer1_Drug_Holdout_Feasibility.csv


In [11]:
# ============================================================
# STEP 8 — Find valid repeated 20% drug-holdout splits
# ============================================================

HOLDOUT_FRACTION = 0.20
N_REQUIRED_SPLITS = 5

valid_splits = []

for seed in range(1, 501):

    rng = np.random.default_rng(seed)

    n_unseen = int(round(
        len(all_drugs_array) * HOLDOUT_FRACTION
    ))

    unseen = set(
        rng.choice(
            all_drugs_array,
            size=n_unseen,
            replace=False
        )
    )

    d1_unseen = pairs_unique["Drug1"].isin(unseen)
    d2_unseen = pairs_unique["Drug2"].isin(unseen)

    train_mask = (~d1_unseen) & (~d2_unseen)
    one_mask   = d1_unseen ^ d2_unseen
    both_mask  = d1_unseen & d2_unseen

    train_tmp = pairs_unique[train_mask]
    one_tmp   = pairs_unique[one_mask]
    both_tmp  = pairs_unique[both_mask]

    train_classes = set(train_tmp["Interaction_ID"])
    one_classes   = set(one_tmp["Interaction_ID"])
    both_classes  = set(both_tmp["Interaction_ID"])

    # Main validity criterion:
    # all 61 Level-2 classes must remain represented in training
    if len(train_classes) == 61:

        valid_splits.append({
            "Seed": seed,
            "N_unseen_drugs": n_unseen,
            "Train_pairs": len(train_tmp),
            "One_unseen_pairs": len(one_tmp),
            "Both_unseen_pairs": len(both_tmp),
            "Train_classes": len(train_classes),
            "One_unseen_classes": len(one_classes),
            "Both_unseen_classes": len(both_classes),
            "Unseen_drugs": unseen
        })

    if len(valid_splits) == N_REQUIRED_SPLITS:
        break


print("=== VALID 20% DRUG-HOLDOUT SPLITS ===")
print(f"Valid splits found: {len(valid_splits)}")

valid_summary = pd.DataFrame([
    {
        k: v
        for k, v in split.items()
        if k != "Unseen_drugs"
    }
    for split in valid_splits
])

display(valid_summary)

print("\nSelected seeds:")
print(valid_summary["Seed"].tolist())

print("\nMean pair counts:")
print(
    valid_summary[
        ["Train_pairs",
         "One_unseen_pairs",
         "Both_unseen_pairs"]
    ].mean().round(1)
)

=== VALID 20% DRUG-HOLDOUT SPLITS ===
Valid splits found: 5


,Seed,N_unseen_drugs,Train_pairs,One_unseen_pairs,Both_unseen_pairs,Train_classes,One_unseen_classes,Both_unseen_classes
0,10,369,141233,70128,8580,61,61,54
1,11,369,141845,69639,8457,61,61,52
2,12,369,141418,69923,8600,61,61,49
3,14,369,142064,69523,8354,61,61,53
4,17,369,138657,71967,9317,61,61,52



Selected seeds:
[10, 11, 12, 14, 17]

Mean pair counts:
Train_pairs          141043.4
One_unseen_pairs      70236.0
Both_unseen_pairs      8661.6
dtype: float64


In [12]:
# ============================================================
# STEP 9 — Construct clean single-label Level-2 dataset
# ============================================================

# Identify genuine multi-label Pair_IDs
multilabel_ids = set(
    full_data.groupby("Pair_ID")["Interaction_ID"]
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

print("Original records :", f"{len(full_data):,}")
print("Original pairs   :", f"{full_data['Pair_ID'].nunique():,}")
print("Multi-label pairs:", f"{len(multilabel_ids):,}")

# ------------------------------------------------------------
# 1. Remove genuine multi-label pairs entirely
# ------------------------------------------------------------

single_label_data = full_data[
    ~full_data["Pair_ID"].isin(multilabel_ids)
].copy()

print("\nAfter excluding multi-label pairs:")
print("Records:", f"{len(single_label_data):,}")
print("Pairs  :", f"{single_label_data['Pair_ID'].nunique():,}")

# ------------------------------------------------------------
# 2. Remove exact pair-level duplicates
#    (same Pair_ID carrying the same Interaction_ID)
# ------------------------------------------------------------

clean_level2 = (
    single_label_data
    .drop_duplicates(subset=["Pair_ID"])
    .reset_index(drop=True)
)

print("\nAfter removing same-label duplicate pair records:")
print("Final records:", f"{len(clean_level2):,}")
print("Final pairs  :", f"{clean_level2['Pair_ID'].nunique():,}")

# ------------------------------------------------------------
# 3. Integrity checks
# ------------------------------------------------------------

labels_per_pair_clean = (
    clean_level2.groupby("Pair_ID")["Interaction_ID"].nunique()
)

print("\n=== INTEGRITY CHECK ===")
print(
    "Duplicated Pair_ID:",
    clean_level2["Pair_ID"].duplicated().sum()
)

print(
    "Pairs with >1 Interaction_ID:",
    (labels_per_pair_clean > 1).sum()
)

print(
    "Number of Level-2 classes:",
    clean_level2["Interaction_ID"].nunique()
)

print(
    "Unique drugs:",
    len(
        set(clean_level2["Drug1"]).union(
            set(clean_level2["Drug2"])
        )
    )
)

# ------------------------------------------------------------
# 4. Save clean analysis dataset
# ------------------------------------------------------------

clean_file = BASE / "Level2_Clean_SingleLabel_Reviewer1.csv"

clean_level2.to_csv(
    clean_file,
    index=False
)

print("\nSaved:")
print(clean_file)

Original records : 220,434
Original pairs   : 219,941
Multi-label pairs: 423

After excluding multi-label pairs:
Records: 219,588
Pairs  : 219,518

After removing same-label duplicate pair records:
Final records: 219,518
Final pairs  : 219,518

=== INTEGRITY CHECK ===
Duplicated Pair_ID: 0
Pairs with >1 Interaction_ID: 0
Number of Level-2 classes: 61
Unique drugs: 1845

Saved:
C:\Users\user\Desktop\drug drug\drug2\results_level2_pair_disjoint\Level2_Clean_SingleLabel_Reviewer1.csv


In [13]:
# ============================================================
# STEP 10 — Rebuild final 20% drug-holdout splits
#           using CLEAN single-label Level-2 dataset
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

FINAL_HOLDOUT_FRACTION = 0.20
N_FINAL_SPLITS = 5

# All drugs from the clean dataset
clean_drugs = np.array(
    sorted(
        set(clean_level2["Drug1"]).union(
            set(clean_level2["Drug2"])
        )
    )
)

print("=== CLEAN DATASET ===")
print(f"Pairs       : {len(clean_level2):,}")
print(f"Unique drugs: {len(clean_drugs):,}")
print(f"Classes     : {clean_level2['Interaction_ID'].nunique()}")

n_unseen = int(round(
    len(clean_drugs) * FINAL_HOLDOUT_FRACTION
))

print(f"\n20% unseen drugs = {n_unseen}")

final_splits = []

for seed in range(1, 1001):

    rng = np.random.default_rng(seed)

    unseen_drugs = set(
        rng.choice(
            clean_drugs,
            size=n_unseen,
            replace=False
        )
    )

    d1_unseen = clean_level2["Drug1"].isin(unseen_drugs)
    d2_unseen = clean_level2["Drug2"].isin(unseen_drugs)

    # Neither drug unseen
    train_mask = (~d1_unseen) & (~d2_unseen)

    # Exactly one drug unseen
    one_mask = d1_unseen ^ d2_unseen

    # Both drugs unseen
    both_mask = d1_unseen & d2_unseen

    train_split = clean_level2.loc[train_mask].copy()
    one_split   = clean_level2.loc[one_mask].copy()
    both_split  = clean_level2.loc[both_mask].copy()

    # Validity criterion:
    # all 61 classes must remain available for model training
    if train_split["Interaction_ID"].nunique() != 61:
        continue

    final_splits.append({
        "Seed": seed,
        "N_unseen_drugs": len(unseen_drugs),

        "Train_pairs": len(train_split),
        "One_unseen_pairs": len(one_split),
        "Both_unseen_pairs": len(both_split),

        "Train_classes": train_split["Interaction_ID"].nunique(),
        "One_unseen_classes": one_split["Interaction_ID"].nunique(),
        "Both_unseen_classes": both_split["Interaction_ID"].nunique(),

        "Unseen_drugs": unseen_drugs,

        "Train_data": train_split,
        "One_unseen_data": one_split,
        "Both_unseen_data": both_split
    })

    if len(final_splits) == N_FINAL_SPLITS:
        break


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

final_summary = pd.DataFrame([
    {
        "Seed": s["Seed"],
        "N_unseen_drugs": s["N_unseen_drugs"],
        "Train_pairs": s["Train_pairs"],
        "One_unseen_pairs": s["One_unseen_pairs"],
        "Both_unseen_pairs": s["Both_unseen_pairs"],
        "Train_classes": s["Train_classes"],
        "One_unseen_classes": s["One_unseen_classes"],
        "Both_unseen_classes": s["Both_unseen_classes"]
    }
    for s in final_splits
])

print("\n=== FINAL CLEAN 20% DRUG-HOLDOUT SPLITS ===")
display(final_summary)

print("\nSelected seeds:")
print(final_summary["Seed"].tolist())

print("\nMean sizes:")
display(
    final_summary[
        [
            "Train_pairs",
            "One_unseen_pairs",
            "Both_unseen_pairs"
        ]
    ].mean().round(1).to_frame("Mean")
)

# Save summary
final_summary.to_csv(
    BASE / "Reviewer1_Final_DrugHoldout_Splits_Summary.csv",
    index=False
)

print("\nSaved: Reviewer1_Final_DrugHoldout_Splits_Summary.csv")

=== CLEAN DATASET ===
Pairs       : 219,518
Unique drugs: 1,845
Classes     : 61

20% unseen drugs = 369

=== FINAL CLEAN 20% DRUG-HOLDOUT SPLITS ===


,Seed,N_unseen_drugs,Train_pairs,One_unseen_pairs,Both_unseen_pairs,Train_classes,One_unseen_classes,Both_unseen_classes
0,10,369,140903,70038,8577,61,61,54
1,11,369,141507,69559,8452,61,61,52
2,12,369,141113,69817,8588,61,61,49
3,14,369,141855,69345,8318,61,61,53
4,17,369,138363,71844,9311,61,61,52



Selected seeds:
[10, 11, 12, 14, 17]

Mean sizes:


,Mean
Train_pairs,140748.2
One_unseen_pairs,70120.6
Both_unseen_pairs,8649.2



Saved: Reviewer1_Final_DrugHoldout_Splits_Summary.csv


In [14]:
# ============================================================
# STEP 11 — Save the 5 final repeated drug-holdout splits
# ============================================================

SPLIT_DIR = BASE / "reviewer1_drug_disjoint_splits"
SPLIT_DIR.mkdir(exist_ok=True)

saved_summary = []

for split in final_splits:

    seed = split["Seed"]

    seed_dir = SPLIT_DIR / f"seed_{seed}"
    seed_dir.mkdir(exist_ok=True)

    train_df = split["Train_data"]
    one_df   = split["One_unseen_data"]
    both_df  = split["Both_unseen_data"]

    # Save datasets
    train_df.to_csv(
        seed_dir / f"Train_seed{seed}.csv",
        index=False
    )

    one_df.to_csv(
        seed_dir / f"OneUnseen_seed{seed}.csv",
        index=False
    )

    both_df.to_csv(
        seed_dir / f"FullyDrugDisjoint_seed{seed}.csv",
        index=False
    )

    # Save unseen drug identities
    pd.DataFrame({
        "Unseen_Drug": sorted(split["Unseen_drugs"])
    }).to_csv(
        seed_dir / f"UnseenDrugs_seed{seed}.csv",
        index=False
    )

    # --------------------------------------------------------
    # Strict integrity checks
    # --------------------------------------------------------

    train_drugs = set(train_df["Drug1"]).union(train_df["Drug2"])
    one_drugs_1 = one_df["Drug1"].isin(split["Unseen_drugs"])
    one_drugs_2 = one_df["Drug2"].isin(split["Unseen_drugs"])

    both_drugs_1 = both_df["Drug1"].isin(split["Unseen_drugs"])
    both_drugs_2 = both_df["Drug2"].isin(split["Unseen_drugs"])

    # No unseen drug is allowed in training
    unseen_in_train = train_drugs.intersection(
        split["Unseen_drugs"]
    )

    # Exactly one unseen drug in OneUnseen
    one_valid = (
        (one_drugs_1 ^ one_drugs_2).all()
    )

    # Both must be unseen in FullyDrugDisjoint
    both_valid = (
        (both_drugs_1 & both_drugs_2).all()
    )

    # Pair IDs must not overlap
    train_ids = set(train_df["Pair_ID"])
    one_ids   = set(one_df["Pair_ID"])
    both_ids  = set(both_df["Pair_ID"])

    pair_overlap = (
        len(train_ids & one_ids) +
        len(train_ids & both_ids) +
        len(one_ids & both_ids)
    )

    saved_summary.append({
        "Seed": seed,
        "Train_N": len(train_df),
        "OneUnseen_N": len(one_df),
        "FullyDisjoint_N": len(both_df),
        "Unseen_in_train": len(unseen_in_train),
        "OneUnseen_valid": one_valid,
        "FullyDisjoint_valid": both_valid,
        "Pair_overlap": pair_overlap
    })


integrity = pd.DataFrame(saved_summary)

print("=== SAVED SPLITS — INTEGRITY AUDIT ===")
display(integrity)

integrity.to_csv(
    SPLIT_DIR / "Split_Integrity_Audit.csv",
    index=False
)

print("\nFiles saved under:")
print(SPLIT_DIR)

=== SAVED SPLITS — INTEGRITY AUDIT ===


,Seed,Train_N,OneUnseen_N,FullyDisjoint_N,Unseen_in_train,OneUnseen_valid,FullyDisjoint_valid,Pair_overlap
0,10,140903,70038,8577,0,True,True,0
1,11,141507,69559,8452,0,True,True,0
2,12,141113,69817,8588,0,True,True,0
3,14,141855,69345,8318,0,True,True,0
4,17,138363,71844,9311,0,True,True,0



Files saved under:
C:\Users\user\Desktop\drug drug\drug2\results_level2_pair_disjoint\reviewer1_drug_disjoint_splits


In [3]:
from pathlib import Path

BASE = Path.cwd()

SPLIT_DIR = BASE / "reviewer1_drug_disjoint_splits"

print("BASE      :", BASE)
print("SPLIT_DIR :", SPLIT_DIR)
print("Exists    :", SPLIT_DIR.exists())

BASE      : C:\Users\user\Desktop\drug drug\drug2\results_level2_pair_disjoint
SPLIT_DIR : C:\Users\user\Desktop\drug drug\drug2\results_level2_pair_disjoint\reviewer1_drug_disjoint_splits
Exists    : True


In [4]:
# ============================================================
# STEP 12 — Prepare Morgan features for FINAL Seed 10
# Same representation as original Level 2 model
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

SEED = 10

seed_dir = SPLIT_DIR / f"seed_{SEED}"

train_seed = pd.read_csv(
    seed_dir / f"Train_seed{SEED}.csv"
)

one_seed = pd.read_csv(
    seed_dir / f"OneUnseen_seed{SEED}.csv"
)

both_seed = pd.read_csv(
    seed_dir / f"FullyDrugDisjoint_seed{SEED}.csv"
)

print("=== SEED 10 DATA ===")
print("Train              :", train_seed.shape)
print("One unseen         :", one_seed.shape)
print("Fully drug-disjoint:", both_seed.shape)

# ============================================================
# Morgan parameters — IDENTICAL TO ORIGINAL LEVEL 2
# ============================================================

MORGAN_RADIUS = 2
MORGAN_BITS = 2048
PAIR_FEATURES = 4096

print("\n=== MORGAN PARAMETERS ===")
print("Radius       :", MORGAN_RADIUS)
print("Bits/drug    :", MORGAN_BITS)
print("Pair features:", PAIR_FEATURES)


def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=MORGAN_RADIUS,
        nBits=MORGAN_BITS
    )

    arr = np.zeros(
        (MORGAN_BITS,),
        dtype=np.uint8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    return arr


# ============================================================
# Build ONE fingerprint cache from all drugs
# ============================================================

all_seed_data = pd.concat(
    [
        train_seed,
        one_seed,
        both_seed
    ],
    ignore_index=True
)

drug_smiles = pd.concat(
    [
        all_seed_data[
            ["Drug1", "SMILES1"]
        ].rename(
            columns={
                "Drug1": "Drug",
                "SMILES1": "SMILES"
            }
        ),

        all_seed_data[
            ["Drug2", "SMILES2"]
        ].rename(
            columns={
                "Drug2": "Drug",
                "SMILES2": "SMILES"
            }
        )
    ],
    ignore_index=True
)

drug_smiles = (
    drug_smiles
    .drop_duplicates("Drug")
    .reset_index(drop=True)
)

drug_fp_cache = {}
invalid_drugs = []

for row in tqdm(
    drug_smiles.itertuples(index=False),
    total=len(drug_smiles),
    desc="Morgan fingerprints"
):

    fp = smiles_to_morgan(row.SMILES)

    if fp is None:
        invalid_drugs.append(
            {
                "Drug": row.Drug,
                "SMILES": row.SMILES
            }
        )
    else:
        drug_fp_cache[row.Drug] = fp


print("\nValid drug fingerprints:", len(drug_fp_cache))
print("Invalid drugs          :", len(invalid_drugs))


# ============================================================
# Pair feature builder
# ============================================================

def build_pair_matrix(df):

    rows = []
    labels = []
    valid_indices = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Building pair matrix"
    ):

        fp1 = drug_fp_cache.get(row["Drug1"])
        fp2 = drug_fp_cache.get(row["Drug2"])

        if fp1 is None or fp2 is None:
            continue

        pair_fp = np.concatenate(
            [fp1, fp2]
        )

        rows.append(pair_fp)
        labels.append(row["Interaction_ID"])
        valid_indices.append(idx)

    X = sparse.csr_matrix(
        np.asarray(
            rows,
            dtype=np.uint8
        )
    )

    y_raw = np.asarray(labels)

    valid_df = (
        df.loc[valid_indices]
        .copy()
        .reset_index(drop=True)
    )

    return X, y_raw, valid_df


X_train_10, y_train_raw_10, train_valid_10 = (
    build_pair_matrix(train_seed)
)

X_one_10, y_one_raw_10, one_valid_10 = (
    build_pair_matrix(one_seed)
)

X_both_10, y_both_raw_10, both_valid_10 = (
    build_pair_matrix(both_seed)
)


# ============================================================
# Label encoding based ONLY on training classes
# ============================================================

le_10 = LabelEncoder()

y_train_10 = le_10.fit_transform(
    y_train_raw_10
)

class_to_encoded = {
    cls: idx
    for idx, cls in enumerate(le_10.classes_)
}

# Both evaluation sets can contain only classes known in training.
assert set(y_one_raw_10).issubset(
    set(le_10.classes_)
)

assert set(y_both_raw_10).issubset(
    set(le_10.classes_)
)

y_one_10 = np.array(
    [
        class_to_encoded[x]
        for x in y_one_raw_10
    ]
)

y_both_10 = np.array(
    [
        class_to_encoded[x]
        for x in y_both_raw_10
    ]
)


print("\n=== FINAL FEATURE MATRICES ===")
print("Train X :", X_train_10.shape)
print("One X   :", X_one_10.shape)
print("Both X  :", X_both_10.shape)

print("\nTraining classes:", len(le_10.classes_))
print(
    "One-unseen observed classes:",
    len(np.unique(y_one_10))
)
print(
    "Fully-unseen observed classes:",
    len(np.unique(y_both_10))
)

print("\n✓ Seed 10 ready for XGBoost.")

=== SEED 10 DATA ===
Train              : (140903, 8)
One unseen         : (70038, 8)
Fully drug-disjoint: (8577, 8)

=== MORGAN PARAMETERS ===
Radius       : 2
Bits/drug    : 2048
Pair features: 4096


Morgan fingerprints:   0%|          | 0/1845 [00:00<?, ?it/s]

[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerator
[20:05:48] DEPRECATION WARNING: please use MorganGenerat


Valid drug fingerprints: 1844
Invalid drugs          : 1


[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerator
[20:05:49] DEPRECATION WARNING: please use MorganGenerat

Building pair matrix:   0%|          | 0/140903 [00:00<?, ?it/s]

Building pair matrix:   0%|          | 0/70038 [00:00<?, ?it/s]

Building pair matrix:   0%|          | 0/8577 [00:00<?, ?it/s]


=== FINAL FEATURE MATRICES ===
Train X : (140895, 4096)
One X   : (70037, 4096)
Both X  : (8577, 4096)

Training classes: 61
One-unseen observed classes: 61
Fully-unseen observed classes: 54

✓ Seed 10 ready for XGBoost.


In [5]:
# ============================================================
# STEP 13 — Train original Level-2 XGBoost on Seed 10
# Evaluate One-Unseen and Fully Drug-Disjoint
# ============================================================

import time
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef
)

# ------------------------------------------------------------
# Original Level-2 XGBoost specification
# ------------------------------------------------------------

model_10 = XGBClassifier(
    objective="multi:softprob",
    num_class=61,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

print("=== TRAINING SEED 10 ===")

t0 = time.time()

model_10.fit(
    X_train_10,
    y_train_10
)

elapsed = time.time() - t0

print(f"Training completed in {elapsed/60:.2f} min")


# ============================================================
# Evaluation function
# ============================================================

def evaluate_level2(model, X, y, name):

    pred = model.predict(X)
    proba = model.predict_proba(X)

    accuracy = accuracy_score(y, pred)

    balanced = balanced_accuracy_score(
        y,
        pred
    )

    macro_f1 = f1_score(
        y,
        pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y,
        pred,
        average="weighted",
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y,
        pred
    )

    # Top-3 accuracy
    top3_idx = np.argsort(
        proba,
        axis=1
    )[:, -3:]

    top3 = np.mean([
        true_label in row
        for true_label, row
        in zip(y, top3_idx)
    ])

    result = {
        "Evaluation": name,
        "N": len(y),
        "Observed_classes": len(np.unique(y)),
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced,
        "Macro_F1": macro_f1,
        "Weighted_F1": weighted_f1,
        "MCC": mcc,
        "Top3_Accuracy": top3
    }

    return result, pred, proba


# ============================================================
# ONE-UNSEEN-DRUG
# ============================================================

one_result_10, pred_one_10, proba_one_10 = (
    evaluate_level2(
        model_10,
        X_one_10,
        y_one_10,
        "One-unseen-drug"
    )
)


# ============================================================
# FULLY DRUG-DISJOINT
# ============================================================

both_result_10, pred_both_10, proba_both_10 = (
    evaluate_level2(
        model_10,
        X_both_10,
        y_both_10,
        "Fully drug-disjoint"
    )
)


# ============================================================
# Results
# ============================================================

results_seed10 = pd.DataFrame([
    one_result_10,
    both_result_10
])

print("\n=== SEED 10 RESULTS ===")

display(
    results_seed10.style.format({
        "Accuracy": "{:.4f}",
        "Balanced_Accuracy": "{:.4f}",
        "Macro_F1": "{:.4f}",
        "Weighted_F1": "{:.4f}",
        "MCC": "{:.4f}",
        "Top3_Accuracy": "{:.4f}"
    })
)


# ============================================================
# Save
# ============================================================

results_seed10.to_csv(
    BASE / "Reviewer1_Seed10_DrugDisjoint_Performance.csv",
    index=False
)

print(
    "\nSaved: Reviewer1_Seed10_DrugDisjoint_Performance.csv"
)

=== TRAINING SEED 10 ===
Training completed in 6.09 min

=== SEED 10 RESULTS ===


,Evaluation,N,Observed_classes,Accuracy,Balanced_Accuracy,Macro_F1,Weighted_F1,MCC,Top3_Accuracy
0,One-unseen-drug,70037,61,0.6606,0.6245,0.6771,0.6492,0.5974,0.8772
1,Fully drug-disjoint,8577,54,0.4695,0.3107,0.3642,0.4198,0.3366,0.7394



Saved: Reviewer1_Seed10_DrugDisjoint_Performance.csv


In [6]:
# ============================================================
# STEP 14 — Run remaining repeated drug-holdout evaluations
# Seeds: 11, 12, 14, 17
# Same Morgan representation + same original XGBoost
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef
)
from xgboost import XGBClassifier
from tqdm.auto import tqdm


REMAINING_SEEDS = [11, 12, 14, 17]

all_results = []

# Keep Seed 10 result already obtained
seed10_copy = results_seed10.copy()
seed10_copy.insert(0, "Seed", 10)
all_results.append(seed10_copy)


# ============================================================
# Reusable pair-matrix builder
# Uses existing drug_fp_cache when possible
# ============================================================

def build_matrix_for_seed(df, fp_cache):

    rows = []
    labels = []
    valid_indices = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Building pair matrix",
        leave=False
    ):

        d1 = row["Drug1"]
        d2 = row["Drug2"]

        # Generate fingerprint only if not already cached
        if d1 not in fp_cache:
            fp = smiles_to_morgan(row["SMILES1"])
            if fp is not None:
                fp_cache[d1] = fp

        if d2 not in fp_cache:
            fp = smiles_to_morgan(row["SMILES2"])
            if fp is not None:
                fp_cache[d2] = fp

        fp1 = fp_cache.get(d1)
        fp2 = fp_cache.get(d2)

        if fp1 is None or fp2 is None:
            continue

        rows.append(
            np.concatenate([fp1, fp2])
        )

        labels.append(
            row["Interaction_ID"]
        )

        valid_indices.append(idx)

    X = sparse.csr_matrix(
        np.asarray(rows, dtype=np.uint8)
    )

    y_raw = np.asarray(labels)

    valid_df = (
        df.loc[valid_indices]
        .copy()
        .reset_index(drop=True)
    )

    return X, y_raw, valid_df


# ============================================================
# Evaluation
# ============================================================

def evaluate_repeated(model, X, y, name):

    pred = model.predict(X)
    proba = model.predict_proba(X)

    top3_idx = np.argsort(
        proba,
        axis=1
    )[:, -3:]

    top3 = np.mean([
        true_label in row
        for true_label, row in zip(y, top3_idx)
    ])

    return {
        "Evaluation": name,
        "N": len(y),
        "Observed_classes": len(np.unique(y)),
        "Accuracy": accuracy_score(y, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y, pred),
        "Macro_F1": f1_score(
            y, pred,
            average="macro",
            zero_division=0
        ),
        "Weighted_F1": f1_score(
            y, pred,
            average="weighted",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(y, pred),
        "Top3_Accuracy": top3
    }


# ============================================================
# Run seeds
# ============================================================

for SEED in REMAINING_SEEDS:

    print("\n" + "=" * 70)
    print(f"STARTING SEED {SEED}")
    print("=" * 70)

    seed_dir = SPLIT_DIR / f"seed_{SEED}"

    train_df = pd.read_csv(
        seed_dir / f"Train_seed{SEED}.csv"
    )

    one_df = pd.read_csv(
        seed_dir / f"OneUnseen_seed{SEED}.csv"
    )

    both_df = pd.read_csv(
        seed_dir / f"FullyDrugDisjoint_seed{SEED}.csv"
    )

    print("Train :", len(train_df))
    print("One   :", len(one_df))
    print("Both  :", len(both_df))

    # --------------------------------------------------------
    # Build matrices
    # --------------------------------------------------------

    X_train, y_train_raw, _ = build_matrix_for_seed(
        train_df,
        drug_fp_cache
    )

    X_one, y_one_raw, _ = build_matrix_for_seed(
        one_df,
        drug_fp_cache
    )

    X_both, y_both_raw, _ = build_matrix_for_seed(
        both_df,
        drug_fp_cache
    )

    print("\nMatrices:")
    print("Train:", X_train.shape)
    print("One  :", X_one.shape)
    print("Both :", X_both.shape)

    # --------------------------------------------------------
    # Encode classes from TRAIN only
    # --------------------------------------------------------

    le = LabelEncoder()

    y_train = le.fit_transform(
        y_train_raw
    )

    class_map = {
        cls: i
        for i, cls in enumerate(le.classes_)
    }

    assert len(le.classes_) == 61

    assert set(y_one_raw).issubset(
        set(le.classes_)
    )

    assert set(y_both_raw).issubset(
        set(le.classes_)
    )

    y_one = np.array([
        class_map[x]
        for x in y_one_raw
    ])

    y_both = np.array([
        class_map[x]
        for x in y_both_raw
    ])

    # --------------------------------------------------------
    # EXACT SAME XGBoost
    # --------------------------------------------------------

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=61,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    print(f"\nTraining Seed {SEED}...")

    t0 = time.time()

    model.fit(
        X_train,
        y_train
    )

    minutes = (
        time.time() - t0
    ) / 60

    print(
        f"Training completed: {minutes:.2f} min"
    )

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    one_result = evaluate_repeated(
        model,
        X_one,
        y_one,
        "One-unseen-drug"
    )

    both_result = evaluate_repeated(
        model,
        X_both,
        y_both,
        "Fully drug-disjoint"
    )

    seed_results = pd.DataFrame([
        one_result,
        both_result
    ])

    seed_results.insert(
        0,
        "Seed",
        SEED
    )

    print(f"\n=== SEED {SEED} RESULTS ===")

    display(
        seed_results.style.format({
            "Accuracy": "{:.4f}",
            "Balanced_Accuracy": "{:.4f}",
            "Macro_F1": "{:.4f}",
            "Weighted_F1": "{:.4f}",
            "MCC": "{:.4f}",
            "Top3_Accuracy": "{:.4f}"
        })
    )

    # --------------------------------------------------------
    # Save immediately
    # --------------------------------------------------------

    seed_results.to_csv(
        BASE /
        f"Reviewer1_Seed{SEED}_DrugDisjoint_Performance.csv",
        index=False
    )

    all_results.append(
        seed_results
    )

    combined_so_far = pd.concat(
        all_results,
        ignore_index=True
    )

    combined_so_far.to_csv(
        BASE /
        "Reviewer1_AllSeeds_DrugDisjoint_Performance.csv",
        index=False
    )

    print(
        f"✓ Seed {SEED} saved."
    )

    # --------------------------------------------------------
    # Free memory before next seed
    # --------------------------------------------------------

    del (
        model,
        X_train,
        X_one,
        X_both,
        y_train,
        y_one,
        y_both
    )

    gc.collect()


# ============================================================
# FINAL TABLE — all five seeds
# ============================================================

all_results_df = pd.concat(
    all_results,
    ignore_index=True
)

print("\n")
print("=" * 70)
print("ALL FIVE SEEDS COMPLETED")
print("=" * 70)

display(
    all_results_df.style.format({
        "Accuracy": "{:.4f}",
        "Balanced_Accuracy": "{:.4f}",
        "Macro_F1": "{:.4f}",
        "Weighted_F1": "{:.4f}",
        "MCC": "{:.4f}",
        "Top3_Accuracy": "{:.4f}"
    })
)

all_results_df.to_csv(
    BASE /
    "Reviewer1_AllSeeds_DrugDisjoint_Performance.csv",
    index=False
)

print("\n✓ Final repeated-split results saved.")


STARTING SEED 11
Train : 141507
One   : 69559
Both  : 8452


Building pair matrix:   0%|          | 0/141507 [00:00<?, ?it/s]

[20:20:08] SMILES Parse Error: syntax error while parsing: OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1
[20:20:08] SMILES Parse Error: check for mistakes around position 76:
[20:20:08] C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C
[20:20:08] ~~~~~~~~~~~~~~~~~~~~^
[20:20:08] SMILES Parse Error: extra open parentheses while parsing: OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1
[20:20:08] SMILES Parse Error: check for mistakes around position 32:
[20:20:08] C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2
[20:20:08] ~~~~~~~~~~~~~~~~~~~~^
[20:20:08] SMILES Parse Error: extra open parentheses while parsing: OC1=CC=CC(=C1)C-1=C2\CCC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC(O)=CC=C1)C1=CC(O)=CC=C1)\C1=CC(O)=CC=C1
[20:20:08] SMILES Parse Error: check for mistakes around position 49:
[20:20:08] )\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=

Building pair matrix:   0%|          | 0/69559 [00:00<?, ?it/s]

Building pair matrix:   0%|          | 0/8452 [00:00<?, ?it/s]

...

C:\Users\user\anaconda3\envs\mers_retrain\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,Seed,Evaluation,N,Observed_classes,Accuracy,Balanced_Accuracy,Macro_F1,Weighted_F1,MCC,Top3_Accuracy
0,17,One-unseen-drug,71842,61,0.6867,0.6756,0.7171,0.6774,0.6295,0.8785
1,17,Fully drug-disjoint,9311,52,0.4787,0.3944,0.4337,0.4386,0.3562,0.7407


✓ Seed 17 saved.


ALL FIVE SEEDS COMPLETED


,Seed,Evaluation,N,Observed_classes,Accuracy,Balanced_Accuracy,Macro_F1,Weighted_F1,MCC,Top3_Accuracy
0,10,One-unseen-drug,70037,61,0.6606,0.6245,0.6771,0.6492,0.5974,0.8772
1,10,Fully drug-disjoint,8577,54,0.4695,0.3107,0.3642,0.4198,0.3366,0.7394
2,11,One-unseen-drug,69559,61,0.6544,0.6556,0.6982,0.6433,0.5926,0.8757
3,11,Fully drug-disjoint,8452,52,0.4584,0.3840,0.4189,0.4120,0.3342,0.7444
4,12,One-unseen-drug,69816,61,0.6319,0.6142,0.6455,0.6200,0.5712,0.8591
5,12,Fully drug-disjoint,8588,49,0.4061,0.2624,0.2666,0.3623,0.2967,0.7111
6,14,One-unseen-drug,69343,61,0.6550,0.6101,0.6745,0.6446,0.5977,0.8704
7,14,Fully drug-disjoint,8318,53,0.4669,0.3421,0.3751,0.4232,0.3665,0.7386
8,17,One-unseen-drug,71842,61,0.6867,0.6756,0.7171,0.6774,0.6295,0.8785
9,17,Fully drug-disjoint,9311,52,0.4787,0.3944,0.4337,0.4386,0.3562,0.7407



✓ Final repeated-split results saved.


In [7]:
# ============================================================
# STEP 15 — Mean ± SD and 95% CI across the 5 repeated splits
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import t

# Use saved results to make this step reproducible
results_file = BASE / "Reviewer1_AllSeeds_DrugDisjoint_Performance.csv"
df = pd.read_csv(results_file)

print("Rows loaded:", len(df))
print("Seeds:", sorted(df["Seed"].unique()))
print("\nEvaluations:")
print(df["Evaluation"].value_counts())

metrics = [
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Weighted_F1",
    "MCC",
    "Top3_Accuracy"
]

summary_rows = []

for evaluation, group in df.groupby("Evaluation"):

    n_splits = len(group)

    for metric in metrics:

        values = group[metric].astype(float).values

        mean = np.mean(values)
        sd = np.std(values, ddof=1)

        # 95% CI of mean using Student t distribution
        sem = sd / np.sqrt(n_splits)

        tcrit = t.ppf(
            0.975,
            df=n_splits - 1
        )

        ci_low = mean - tcrit * sem
        ci_high = mean + tcrit * sem

        summary_rows.append({
            "Evaluation": evaluation,
            "Metric": metric,
            "N_splits": n_splits,
            "Mean": mean,
            "SD": sd,
            "CI95_Lower": ci_low,
            "CI95_Upper": ci_high
        })


summary = pd.DataFrame(summary_rows)

print("\n=== REPEATED DRUG-HOLDOUT PERFORMANCE ===")

display(
    summary.style.format({
        "Mean": "{:.4f}",
        "SD": "{:.4f}",
        "CI95_Lower": "{:.4f}",
        "CI95_Upper": "{:.4f}"
    })
)

summary.to_csv(
    BASE / "Reviewer1_DrugDisjoint_Mean_SD_95CI.csv",
    index=False
)


# ============================================================
# Manuscript-friendly wide table
# ============================================================

paper_rows = []

for evaluation, group in df.groupby("Evaluation"):

    row = {
        "Evaluation": evaluation,
        "Mean_N": int(round(group["N"].mean())),
        "Observed_classes_range":
            f"{group['Observed_classes'].min()}–"
            f"{group['Observed_classes'].max()}"
    }

    for metric in metrics:

        s = summary[
            (summary["Evaluation"] == evaluation) &
            (summary["Metric"] == metric)
        ].iloc[0]

        row[metric] = (
            f"{s['Mean']:.4f} ± {s['SD']:.4f} "
            f"(95% CI {s['CI95_Lower']:.4f}–"
            f"{s['CI95_Upper']:.4f})"
        )

    paper_rows.append(row)


paper_table = pd.DataFrame(paper_rows)

print("\n=== MANUSCRIPT-READY TABLE ===")
display(paper_table)

paper_table.to_csv(
    BASE / "Reviewer1_DrugDisjoint_Manuscript_Table.csv",
    index=False
)

print("\n✓ Mean, SD and 95% CI saved.")

Rows loaded: 10
Seeds: [np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(17)]

Evaluations:
Evaluation
One-unseen-drug        5
Fully drug-disjoint    5
Name: count, dtype: int64

=== REPEATED DRUG-HOLDOUT PERFORMANCE ===


,Evaluation,Metric,N_splits,Mean,SD,CI95_Lower,CI95_Upper
0,Fully drug-disjoint,Accuracy,5,0.4559,0.0288,0.4202,0.4916
1,Fully drug-disjoint,Balanced_Accuracy,5,0.3387,0.0543,0.2714,0.4061
2,Fully drug-disjoint,Macro_F1,5,0.3717,0.0655,0.2903,0.4531
3,Fully drug-disjoint,Weighted_F1,5,0.4112,0.0290,0.3752,0.4471
4,Fully drug-disjoint,MCC,5,0.3380,0.0268,0.3048,0.3713
5,Fully drug-disjoint,Top3_Accuracy,5,0.7349,0.0135,0.7181,0.7516
6,One-unseen-drug,Accuracy,5,0.6577,0.0196,0.6334,0.6820
7,One-unseen-drug,Balanced_Accuracy,5,0.6360,0.0284,0.6007,0.6713
8,One-unseen-drug,Macro_F1,5,0.6825,0.0269,0.6490,0.7159
9,One-unseen-drug,Weighted_F1,5,0.6469,0.0205,0.6215,0.6723



=== MANUSCRIPT-READY TABLE ===


,Evaluation,Mean_N,Observed_classes_range,Accuracy,Balanced_Accuracy,Macro_F1,Weighted_F1,MCC,Top3_Accuracy
0,Fully drug-disjoint,8649,49–54,0.4559 ± 0.0288 (95% CI 0.4202–0.4916),0.3387 ± 0.0543 (95% CI 0.2714–0.4061),0.3717 ± 0.0655 (95% CI 0.2903–0.4531),0.4112 ± 0.0290 (95% CI 0.3752–0.4471),0.3380 ± 0.0268 (95% CI 0.3048–0.3713),0.7349 ± 0.0135 (95% CI 0.7181–0.7516)
1,One-unseen-drug,70119,61–61,0.6577 ± 0.0196 (95% CI 0.6334–0.6820),0.6360 ± 0.0284 (95% CI 0.6007–0.6713),0.6825 ± 0.0269 (95% CI 0.6490–0.7159),0.6469 ± 0.0205 (95% CI 0.6215–0.6723),0.5977 ± 0.0208 (95% CI 0.5718–0.6235),0.8722 ± 0.0079 (95% CI 0.8623–0.8820)



✓ Mean, SD and 95% CI saved.
